In [1]:
import subprocess
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS, PAPILO_PATH
from tools.parse_papilo_log import parse, to_metadata

PAPILO_OUT = Path("/data/energy-system-preprocessing/presolve/papilo")
PAPILO_OUT.mkdir(parents=True, exist_ok=True)


def run_papilo(mps_in: Path) -> bool:
    model_dir = mps_in.parent
    rel       = model_dir.relative_to(MODELS)
    out_dir   = PAPILO_OUT / rel
    out_dir.mkdir(parents=True, exist_ok=True)

    reduced   = out_dir / "reduced.mps"
    postsolve = out_dir / "reduced.postsolve"
    log_file  = out_dir / "output.txt"

    if reduced.exists():
        print(f"  skipped (already exists)", flush=True)
        return True

    cmd = [
        str(PAPILO_PATH), "presolve",
        "-f", str(mps_in),
        "-r", str(reduced),
        "-v", str(postsolve),
        "--message.verbosity=4"
    ]

    with log_file.open("w") as log:
        process = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)

    if process.returncode != 0 or not reduced.exists():
        print(f"  FAILED (rc={process.returncode})")
        return False

    rounds, model_path = parse(log_file.read_text().splitlines(keepends=True))
    meta = to_metadata(rounds, model_path)

    src_meta_path = model_dir / "metadata.json"
    existing: dict = json.loads(src_meta_path.read_text()) if src_meta_path.exists() else {}
    existing.update(meta)
    (out_dir / "metadata.json").write_text(json.dumps(existing, indent=2))

    return True


In [ ]:
models = sorted(MODELS.rglob("original.mps"))
print(f"{len(models)} models found under {MODELS}")


859 models found under /data/energy-system-preprocessing/models


In [3]:
failed = []

for i, mps_in in enumerate(models, 1):
    rel = mps_in.parent.relative_to(MODELS)
    print(f"[{i}/{len(models)}] {rel}", flush=True)
    ok = run_papilo(mps_in)
    if not ok:
        failed.append(str(rel))
    print(flush=True)

if failed:
    print(f"\n{len(failed)} model(s) failed: {failed}")
else:
    print(f"\nAll {len(models)} models presolved successfully.")


[1/859] benchmarks/MIPLIB/10teams
  skipped (already exists)

[2/859] benchmarks/MIPLIB/30n20b8
  skipped (already exists)

[3/859] benchmarks/MIPLIB/irish-electricity
  skipped (already exists)

[4/859] r10_res168_f0.0000_t0.0192
  skipped (already exists)

[5/859] r10_res168_f0.0000_t0.0192_cf0.5
  skipped (already exists)

[6/859] r10_res168_f0.0000_t0.0192_cf0.5_mip
  skipped (already exists)

[7/859] r10_res168_f0.0000_t0.0192_cf0.75
  skipped (already exists)

[8/859] r10_res168_f0.0000_t0.0192_mip
  skipped (already exists)

[9/859] r10_res168_f0.0000_t0.0833
  skipped (already exists)

[10/859] r10_res168_f0.0000_t0.0833_cf0.5
  skipped (already exists)

[11/859] r10_res168_f0.0000_t0.0833_cf0.5_mip
  skipped (already exists)

[12/859] r10_res168_f0.0000_t0.0833_cf0.75
  skipped (already exists)

[13/859] r10_res168_f0.0000_t0.0833_mip
  skipped (already exists)

[14/859] r10_res168_f0.0000_t1.0000
  skipped (already exists)

[15/859] r10_res168_f0.0000_t1.0000_cf0.5
  skipped 